### 0. Download AIRTLab Dataset

In [ ]:
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "cnn_lstm_pytorch.py").is_file():
            return candidate
    raise FileNotFoundError("Start the notebook kernel inside the violence-movies project.")


BASE_DIR = find_project_root(Path.cwd())
print(f"Base directory: {BASE_DIR}")


In [ ]:
# From the project root, download AIRTLab with:
# git clone https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos.git data/raw/violence-detection-dataset

### 1. Separate AIRTLab into three classes using validated CSV action names

`gunshot` is high-level violence; `choke` is low-level violence. Unknown or missing names raise errors. Reclassifying existing files requires corrected cached labels; use the repair helper or rebuild before training.


In [ ]:
# Organize AIRTLab clips using the action names in the supplied CSV files.
from pathlib import Path
import sys

BASE_DIR = find_project_root(Path.cwd())
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))

from airtlab_actions import (
    HIGH_LEVEL_LABELS, LOW_LEVEL_LABELS, NON_VIOLENCE_LABELS,
    normalize_actions, read_csv_map, category_for_clip,
    organize_airtlab, validate_prepared_action_labels,
)

ROOT = BASE_DIR / "data" / "raw" / "violence-detection-dataset"
OUT = BASE_DIR / "data" / "processed" / "violence-detection-dataset"
COPY_FILES = True

# Canonical CSV names: gunshot -> high-level; choke -> low-level.
# shoot/stifle aliases are supported. Unknown/missing actions raise before copying.
# Identical stale copies in the other violence folder are removed on rerun.
def main():
    summary = organize_airtlab(ROOT, OUT, copy_files=COPY_FILES)
    print("AIRTLab organization:", summary)
    print("Output at:", OUT)
    print("If labels changed, rebuild cached arrays or use repair_airtlab_action_labels.py.")

if __name__ == "__main__":
    main()


### 2. Divide each video into overlapping chunks with 16 sampled frames at stride 3 and a resolution of 112 x 112 (e.g. a video may have 8 chunks) and store the samples and labels to data\processed\violence-detection-dataset\airtlabDataset

In [ ]:
# Utility functions for AIRTLab memmap preprocessing.

from collections import Counter
from pathlib import Path
import json
import cv2
import numpy as np

AIRT_CLASS_NAMES = ["non-violence", "low-level violence", "high-level violence"]
AIRT_CAMERAS = ["cam1", "cam2"]
FRAMES_PER_CHUNK = 16
FRAME_SAMPLE_STRIDE = 3
CHUNK_OVERLAP = 0.5
FRAME_SIZE = (112, 112)
SAMPLE_SHAPE = (FRAMES_PER_CHUNK, FRAME_SIZE[1], FRAME_SIZE[0], 3)
SAMPLE_DTYPE = np.float32
LABEL_DTYPE = np.int8
VIDEO_DTYPE = np.int32
RAW_FRAMES_PER_CHUNK = 1 + (FRAMES_PER_CHUNK - 1) * FRAME_SAMPLE_STRIDE
CHUNK_FRAME_STEP = max(1, int(round(FRAMES_PER_CHUNK * (1.0 - CHUNK_OVERLAP) * FRAME_SAMPLE_STRIDE)))


def count_strided_chunks(frame_count: int) -> int:
    if frame_count <= 0:
        return 0
    if frame_count < RAW_FRAMES_PER_CHUNK:
        return 1
    return ((frame_count - RAW_FRAMES_PER_CHUNK) // CHUNK_FRAME_STEP) + 1


def iter_chunk_start_frames(frame_count: int):
    if frame_count <= 0:
        return
    if frame_count < RAW_FRAMES_PER_CHUNK:
        yield 0
        return
    max_start = frame_count - RAW_FRAMES_PER_CHUNK
    for start_frame in range(0, max_start + 1, CHUNK_FRAME_STEP):
        yield start_frame


def sampled_frame_indices(start_frame: int, frame_count: int) -> list[int]:
    last_frame = max(0, frame_count - 1)
    return [min(start_frame + offset * FRAME_SAMPLE_STRIDE, last_frame) for offset in range(FRAMES_PER_CHUNK)]


def iter_airtlab_videos(video_base_path: Path):
    video_base_path = Path(video_base_path)
    for label, class_name in enumerate(AIRT_CLASS_NAMES):
        for camera in AIRT_CAMERAS:
            camera_dir = video_base_path / class_name / camera
            if not camera_dir.exists():
                print(f"[WARN] Missing folder: {camera_dir}")
                continue
            for video_path in sorted(camera_dir.glob("*.mp4")):
                yield label, class_name, camera, video_path


def scan_airtlab_videos(video_base_path: Path) -> list[dict]:
    records = []
    for video_id, (label, class_name, camera, video_path) in enumerate(iter_airtlab_videos(video_base_path)):
        capture = cv2.VideoCapture(str(video_path))
        opened = capture.isOpened()
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        capture.release()
        if not opened or frame_count <= 0:
            raise ValueError(f"Cannot read video: {video_path}")
        records.append(
            {
                "path": video_path,
                "label": label,
                "class_name": class_name,
                "camera": camera,
                "video_id": video_id,
                "frame_count": frame_count,
                "fps": fps,
                "chunk_count": count_strided_chunks(frame_count),
            }
        )
    return records


def count_chunks(videoBasePath):
    return sum(record["chunk_count"] for record in scan_airtlab_videos(Path(videoBasePath)))


def print_airtlab_scan_summary(records: list[dict]) -> None:
    print(f"Videos: {len(records)}")
    print(f"Chunks: {sum(record['chunk_count'] for record in records)}")
    print(
        f"Chunk sampling: {FRAMES_PER_CHUNK} frames, stride={FRAME_SAMPLE_STRIDE}, "
        f"overlap={CHUNK_OVERLAP:.0%}, raw span={RAW_FRAMES_PER_CHUNK} frames, step={CHUNK_FRAME_STEP} frames"
    )
    by_class = Counter(record["class_name"] for record in records)
    chunks_by_class = Counter()
    for record in records:
        chunks_by_class[record["class_name"]] += record["chunk_count"]
    for class_name in AIRT_CLASS_NAMES:
        print(f"  {class_name}: {by_class[class_name]} videos, {chunks_by_class[class_name]} chunks")


def write_airtlab_manifest(feature_base_path: Path, records: list[dict]) -> None:
    payload = {
        "class_names": AIRT_CLASS_NAMES,
        "cameras": AIRT_CAMERAS,
        "frames_per_chunk": FRAMES_PER_CHUNK,
        "frame_sample_stride": FRAME_SAMPLE_STRIDE,
        "chunk_overlap": CHUNK_OVERLAP,
        "raw_frames_per_chunk": RAW_FRAMES_PER_CHUNK,
        "chunk_frame_step": CHUNK_FRAME_STEP,
        "short_video_padding": "repeat_last_frame",
        "frame_size": list(FRAME_SIZE),
        "sample_shape": list(SAMPLE_SHAPE),
        "sample_dtype": np.dtype(SAMPLE_DTYPE).name,
        "label_dtype": np.dtype(LABEL_DTYPE).name,
        "video_dtype": np.dtype(VIDEO_DTYPE).name,
        "total_videos": len(records),
        "total_chunks": sum(record["chunk_count"] for record in records),
        "videos": [
            {
                **record,
                "path": str(record["path"]),
            }
            for record in records
        ],
    }
    (feature_base_path / "mmap_manifest.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")


def read_resized_frames(video: cv2.VideoCapture) -> list[np.ndarray]:
    frames = []
    while True:
        ret, img = video.read()
        if not ret:
            break
        frames.append(cv2.resize(img, FRAME_SIZE))
    return frames


def preprocessVideos(videoBasePath, featureBasePath, verbose=True):
    video_base_path = Path(videoBasePath)
    feature_base_path = Path(featureBasePath)
    feature_base_path.mkdir(parents=True, exist_ok=True)

    records = scan_airtlab_videos(video_base_path)
    total_chunks = sum(record["chunk_count"] for record in records)
    if total_chunks == 0:
        raise ValueError(f"No chunks found under {video_base_path}")

    if verbose:
        print_airtlab_scan_summary(records)

    write_airtlab_manifest(feature_base_path, records)
    metadata_path = feature_base_path / "chunk_metadata.jsonl"

    npSamples = np.memmap(
        feature_base_path / "samples.mmap",
        dtype=SAMPLE_DTYPE,
        mode="w+",
        shape=(total_chunks, *SAMPLE_SHAPE),
    )
    npLabels = np.memmap(feature_base_path / "labels.mmap", dtype=LABEL_DTYPE, mode="w+", shape=(total_chunks,))
    npVideos = np.memmap(feature_base_path / "videos.mmap", dtype=VIDEO_DTYPE, mode="w+", shape=(total_chunks,))

    cnt = 0
    with metadata_path.open("w", encoding="utf-8") as metadata_file:
        for index, record in enumerate(records, start=1):
            video = cv2.VideoCapture(str(record["path"]))
            if not video.isOpened():
                raise ValueError(f"Cannot open video: {record['path']}")

            frames = read_resized_frames(video)
            video.release()

            written = 0
            for chunk_index, start_frame in enumerate(iter_chunk_start_frames(record["frame_count"])):
                frame_indices = sampled_frame_indices(start_frame, len(frames))

                npSamples[cnt] = np.asarray([frames[i] for i in frame_indices], dtype=SAMPLE_DTYPE)
                npLabels[cnt] = record["label"]
                npVideos[cnt] = record["video_id"]
                metadata_file.write(
                    json.dumps(
                        {
                            "sample_index": cnt,
                            "video_id": record["video_id"],
                            "video_path": str(record["path"]),
                            "class_name": record["class_name"],
                            "label": record["label"],
                            "camera": record["camera"],
                            "chunk_index": chunk_index,
                            "start_frame": frame_indices[0],
                            "end_frame": frame_indices[-1],
                            "sampled_frame_indices": frame_indices,
                            "start_time_seconds": frame_indices[0] / record["fps"] if record["fps"] > 0 else None,
                            "end_time_seconds": frame_indices[-1] / record["fps"] if record["fps"] > 0 else None,
                        }
                    )
                    + "\n"
                )
                cnt += 1
                written += 1

            if written != record["chunk_count"]:
                raise RuntimeError(f"Decoded {written} chunks from {record['path']}, expected {record['chunk_count']}")

            if verbose and (index == 1 or index % 25 == 0 or index == len(records)):
                print(f"Processed {index}/{len(records)} videos; wrote {cnt}/{total_chunks} chunks", flush=True)
                npSamples.flush()
                npLabels.flush()
                npVideos.flush()

    if cnt != total_chunks:
        raise RuntimeError(f"Wrote {cnt} chunks, expected {total_chunks}")

    npSamples.flush()
    npLabels.flush()
    npVideos.flush()
    del npSamples, npLabels, npVideos


def validate_airtlab_memmaps(featureBasePath, records: list[dict] | None = None):
    feature_base_path = Path(featureBasePath)
    if records is None:
        records = scan_airtlab_videos(feature_base_path)
    total_chunks = sum(record["chunk_count"] for record in records)

    samples_path = feature_base_path / "samples.mmap"
    labels_path = feature_base_path / "labels.mmap"
    videos_path = feature_base_path / "videos.mmap"
    metadata_path = feature_base_path / "chunk_metadata.jsonl"
    expected_sample_bytes = total_chunks * int(np.prod(SAMPLE_SHAPE)) * np.dtype(SAMPLE_DTYPE).itemsize
    expected_label_bytes = total_chunks * np.dtype(LABEL_DTYPE).itemsize
    expected_video_bytes = total_chunks * np.dtype(VIDEO_DTYPE).itemsize
    expected_sizes = {
        samples_path: expected_sample_bytes,
        labels_path: expected_label_bytes,
        videos_path: expected_video_bytes,
    }
    for mmap_path, expected_size in expected_sizes.items():
        actual_size = mmap_path.stat().st_size
        if actual_size != expected_size:
            raise ValueError(f"{mmap_path} has {actual_size} bytes, expected {expected_size}")

    if metadata_path.exists():
        metadata_count = sum(1 for _ in metadata_path.open("r", encoding="utf-8"))
        if metadata_count != total_chunks:
            raise ValueError(f"{metadata_path} has {metadata_count} rows, expected {total_chunks}")
    else:
        metadata_count = 0

    samples = np.memmap(samples_path, dtype=SAMPLE_DTYPE, mode="r", shape=(total_chunks, *SAMPLE_SHAPE))
    labels = np.memmap(labels_path, dtype=LABEL_DTYPE, mode="r", shape=(total_chunks,))
    videos = np.memmap(videos_path, dtype=VIDEO_DTYPE, mode="r", shape=(total_chunks,))
    unique_labels, label_counts = np.unique(np.asarray(labels), return_counts=True)
    observed_video_ids = set(np.unique(np.asarray(videos)).astype(int).tolist())
    expected_video_ids = {record["video_id"] for record in records if record["chunk_count"] > 0}
    if observed_video_ids != expected_video_ids:
        missing_ids = sorted(expected_video_ids - observed_video_ids)
        unexpected_ids = sorted(observed_video_ids - expected_video_ids)
        raise ValueError(
            f"Video ID mismatch: missing chunked IDs={missing_ids[:10]}, unexpected IDs={unexpected_ids[:10]}"
        )

    print("AIRTLab memmap validation passed")
    print(f"  samples shape: {samples.shape}")
    print(f"  labels shape: {labels.shape}")
    print(f"  videos shape: {videos.shape}")
    print(f"  label distribution: {dict(zip(unique_labels.tolist(), label_counts.tolist()))}")
    print(f"  unique video IDs with chunks: {len(observed_video_ids)} / {len(records)} source videos")
    print(f"  chunk metadata rows: {metadata_count}")
    print(f"  first sample min/max: {float(samples[0].min()):.1f}/{float(samples[0].max()):.1f}")


In [ ]:
OUT = BASE_DIR / "data" / "processed" / "violence-detection-dataset"
REBUILD_AIRTLAB_MMAPS = False

airtlab_records = scan_airtlab_videos(OUT)
print_airtlab_scan_summary(airtlab_records)

mmap_paths = [OUT / name for name in ("samples.mmap", "labels.mmap", "videos.mmap", "mmap_manifest.json", "chunk_metadata.jsonl")]
if REBUILD_AIRTLAB_MMAPS or not all(path.exists() for path in mmap_paths):
    preprocessVideos(OUT, OUT, verbose=True)
else:
    print("Existing AIRTLab memmaps found; validating without rebuilding.")

# Reject stale cached labels after an action-name or class-folder correction.
from airtlab_actions import validate_prepared_action_labels
print(validate_prepared_action_labels(BASE_DIR / "data" / "raw" / "violence-detection-dataset", OUT))
validate_airtlab_memmaps(OUT, airtlab_records)
